# Notes

- Need to run this in the econ-env kernel or else scipy doesn't import.
- All voters and policies are zero indexed.

## Rough Table of Contents

- ~~First, generate a two good case, and test~~
- ~~Second, generate an n goods case. Test using 3 variables, with 5 normal cases and 6 edge ones.~~
- ~~Third, implement code of the paper's CES function~~
- ~~Fourth, test to see if ceteres paribus, our additional parameters actually change the maximization like we want it to (i.e. if our model actually works or needs to be changed)~~
- ~~Fifth, iterate our CES function across all voters, who can select from the appropriate voters (see paper).~~
- ~~Sixth, have voters select policies. Depending on values in our variable preference_matrix (which measures each voter's position on each policy), they then select which policies to vote for. These positions act as alphas for the CES function. Then, how much of each policy they select, we input into the matrix as (voter j's direct voting power) x (voter j's allocation to policy p).~~
- ~~Seventh, output our results into a matrix format, which I can then feed into Yasushi's PPV code.~~
- Eigth, ex ante analysis (i.e. excessive elegation chains, gained influence, all utility calculations )
- Ninth, endogenize

In [ ]:
import numpy as np
import scipy
import random
import pandas as pd

seed_for_prng = 78557
prng = np.random.default_rng(seed_for_prng)

print("NumPy version:", np.__version__)
print("SciPy version:", scipy.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 1.23.5
SciPy version: 1.15.3
Pandas version: 2.3.1


In [33]:
rho = 0.5

## 1.0: Two Good Case

### 1.1: Two Good Model

In [17]:
import numpy as np
from scipy.optimize import linprog

def maximize_utility(util_1, total_budget):
    """
    Solves utility maximization with linear preferences and budget constraint.
    """

    if not (0 <= util_1 <= 1):
        raise ValueError("util_1 must be between 0 and 1.")

    util_2 = 1 - util_1

    # Objective function: maximize U = util_1*x1 + util_2*x2 → minimize -U
    c = [-util_1, -util_2]

    # Budget constraint: x1 + x2 ≤ total_budget
    A_ub = [[1, 1]]
    b_ub = [total_budget]

    # Non-negativity constraints
    bounds = [(0, None), (0, None)]

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

    if result.success:
        x1, x2 = result.x
        utility = util_1 * x1 + util_2 * x2
        return {
            "x1": x1,
            "x2": x2,
            "utility": utility
        }
    else:
        raise RuntimeError("Optimization failed:", result.message)


### 1.2: Testing Two Good Model

In [18]:
test_cases = [
    {"util_1": 0.0, "total_budget": 10},   # All utility from good 2
    {"util_1": 1.0, "total_budget": 10},   # All utility from good 1
    {"util_1": 0.5, "total_budget": 10},   # Indifferent case
    {"util_1": 0.9, "total_budget": 1},    # Small budget, strong preference
    {"util_1": 0.1, "total_budget": 1},    # Small budget, weak preference
    {"util_1": 0.75, "total_budget": 100}, # Strong preference, large budget
    {"util_1": 0.25, "total_budget": 100}, # Opposite bias, large budget
    {"util_1": 0.6, "total_budget": 0.5},  # Very small budget
    {"util_1": 0.4, "total_budget": 1e6},  # Huge budget, low preference
    {"util_1": 0.6, "total_budget": 7.7},  # Arbitrary float budget
]

for i, case in enumerate(test_cases, 1):
    try:
        result = maximize_utility(case["util_1"], case["total_budget"])
        assert all(k in result for k in ['x1', 'x2', 'utility']), "Missing keys in result"
        print(f"Test {i}: util_1={case['util_1']}, budget={case['total_budget']}")
        print(f"  x1 = {result['x1']:.4f}, x2 = {result['x2']:.4f}, utility = {result['utility']:.4f}")
        print("-" * 60)
    except Exception as e:
        print(f"Test {i} FAILED: util_1={case['util_1']}, budget={case['total_budget']}")
        print(f"  Error: {e}")
        print("-" * 60)

Test 1: util_1=0.0, budget=10
  x1 = 0.0000, x2 = 10.0000, utility = 10.0000
------------------------------------------------------------
Test 2: util_1=1.0, budget=10
  x1 = 10.0000, x2 = 0.0000, utility = 10.0000
------------------------------------------------------------
Test 3: util_1=0.5, budget=10
  x1 = 10.0000, x2 = 0.0000, utility = 5.0000
------------------------------------------------------------
Test 4: util_1=0.9, budget=1
  x1 = 1.0000, x2 = 0.0000, utility = 0.9000
------------------------------------------------------------
Test 5: util_1=0.1, budget=1
  x1 = 0.0000, x2 = 1.0000, utility = 0.9000
------------------------------------------------------------
Test 6: util_1=0.75, budget=100
  x1 = 100.0000, x2 = 0.0000, utility = 75.0000
------------------------------------------------------------
Test 7: util_1=0.25, budget=100
  x1 = 0.0000, x2 = 100.0000, utility = 75.0000
------------------------------------------------------------
Test 8: util_1=0.6, budget=0.5
  x1

## 2.0: CES of n Goods

### 2.1: CES of n Goods Model

Function Definition

In [124]:
from scipy.optimize import minimize

def ces_utility(y, alpha, rho):
    y = np.array(y)
    
    # Disallow negative quantities
    if np.any(y < 0):
        return -np.inf
    
    # Guard against undefined behavior when rho < 0 and y == 0
    if rho < 0 and np.any(y == 0):
        return -np.inf
    
    if np.isclose(rho, 0.0):
        # Cobb-Douglas limit
        return np.prod(y ** alpha)
    else:
        return (np.sum(alpha * y**rho))**(1/rho)

def maximize_ces_n_goods(alpha, rho, budget):
    alpha = np.array(alpha)
    n = len(alpha)
    
    # Normalization check
    if not np.isclose(np.sum(alpha), 1.0):
        raise ValueError("Weights alpha must sum to 1.")
    
    # All prices = 1 by assumption
    prices = np.ones(n)
    
    def objective(y):
        return -ces_utility(y, alpha, rho)
    
    # Budget constraint (equality, since CES utility is monotonic)
    constraints = [{'type': 'eq', 'fun': lambda y: budget - np.sum(prices * y)}]
    
    # Non-negativity bounds
    bounds = [(0, None) for _ in range(n)]
    
    # Improved starting point based on alpha (p_i = 1)
    y0 = budget * alpha

    result = minimize(objective, y0, bounds=bounds, constraints=constraints)

    if result.success:
        y_opt = result.x
        utility = ces_utility(y_opt, alpha, rho)
        return {
            'quantities': y_opt,
            'utility': utility
        }
    else:
        raise RuntimeError("Optimization failed: " + result.message)


Function Calling

In [125]:
alpha = [0.4, 0.3, 0.3] #List, this is weighing of each good
rho = 0.0 #Float, this is the exponent of the CES function (i.e. how goods are compared)
budget = 100 #Integer, keep as 100, since all prices are = 1.

result = maximize_ces_n_goods(alpha, rho, budget)
print("Quantities:", np.round(result["quantities"], 4))
print("Utility:", np.round(result["utility"], 4))

Quantities: [40. 30. 30.]
Utility: 33.6587


### 2.2: CES of n Goods Model Testing

Normal cases

In [7]:
def CES_n_goods_test():
    test_cases = [
        {"alpha": [0.4, 0.3, 0.3], "rho": 0.0, "budget": 100},
        {"alpha": [0.5, 0.3, 0.2], "rho": 0.95, "budget": 100},
        {"alpha": [0.5, 0.25, 0.25], "rho": 0.5, "budget": 100},
        {"alpha": [0.6, 0.3, 0.1], "rho": -5, "budget": 100},
        {"alpha": [0.25, 0.25, 0.25, 0.25], "rho": 0.2, "budget": 200},
    ]

    print("Normal Cases")

    for i, case in enumerate(test_cases, 1):
        print(f"\n--- Test Case {i} ---")
        result = maximize_ces_n_goods(case["alpha"], case["rho"], case["budget"])
        print("Quantities:", np.round(result["quantities"], 4))
        print("Utility:", np.round(result["utility"], 4))

    

In [8]:
CES_n_goods_test()

Normal Cases

--- Test Case 1 ---
Quantities: [40. 30. 30.]
Utility: 33.6587

--- Test Case 2 ---
Quantities: [9.99973e+01 2.70000e-03 0.00000e+00]
Utility: 48.2089

--- Test Case 3 ---
Quantities: [66.6666 16.6668 16.6667]
Utility: 37.5

--- Test Case 4 ---
Quantities: [37.9848 33.8405 28.1747]
Utility: 34.6642

--- Test Case 5 ---
Quantities: [50. 50. 50. 50.]
Utility: 50.0


Edge Cases

In [9]:
    # Edge Case 1: alpha does not sum to 1 (should raise ValueError)
print("Edge Case 1: alpha does not sum to 1 (should raise ValueError)")

alpha = [0.5, 0.3, 0.3]  # sum = 1.1
rho = 0.5
budget = 100

result = maximize_ces_n_goods(alpha, rho, budget)
print("Quantities:", np.round(result["quantities"], 4))
print("Utility:", np.round(result["utility"], 4))

Edge Case 1: alpha does not sum to 1 (should raise ValueError)


ValueError: Weights alpha must sum to 1.

In [ ]:
    # Edge Case 2: negative quantity would be optimal (not allowed by bounds)
print("Edge Case 2: negative quantity would be optimal (not allowed by bounds)")

alpha = [0.0, 1.0, 0.0]
rho = -2
budget = 100

result = maximize_ces_n_goods(alpha, rho, budget)
print("Quantities:", np.round(result["quantities"], 4))
print("Utility:", np.round(result["utility"], 4))

In [ ]:
    # Edge Case 3: extremely large positive rho (nearly linear utility)
print("Edge Case 3: extremely large positive rho (nearly linear utility)")

alpha = [0.3, 0.3, 0.4]
rho = 1000
budget = 100


result = maximize_ces_n_goods(alpha, rho, budget)
print("Quantities:", np.round(result["quantities"], 4))
print("Utility:", np.round(result["utility"], 4))

In [ ]:
    # Edge Case 4: very negative rho (strong complementarity, may lead to division by 0 if x_i → 0)
print("Edge Case 4: very negative rho (strong complementarity, may lead to division by 0 if x_i → 0)")

alpha = [0.4, 0.4, 0.2]
rho = -1e6
budget = 100

result = maximize_ces_n_goods(alpha, rho, budget)
print("Quantities:", np.round(result["quantities"], 4))
print("Utility:", np.round(result["utility"], 4))

In [ ]:
    # Edge Case 5: zero budget (no feasible consumption)
print("Edge Case 5: zero budget (no feasible consumption)")

alpha = [0.4, 0.4, 0.2]
rho = 0.3
budget = 0

result = maximize_ces_n_goods(alpha, rho, budget)
print("Quantities:", np.round(result["quantities"], 4))
print("Utility:", np.round(result["utility"], 4))

In [ ]:
    # Edge Case 6: high n value (many goods)
print("Edge Case 6: high n value (many goods)")

alpha = [
    0.1, 0.1, 0.1, 0.1, 0.1, #= 0.5
    0.05, 0.05, 0.05, 0.05, 0.05, # = 0.25
    0.04, 0.04, 0.04, 0.04, 0.04, # = 0.2
    0.01, 0.01, 0.01, 0.01, 0.01 # = .05
]
rho = 0.5
budget = 100

result = maximize_ces_n_goods(alpha, rho, budget)
print("Quantities:", np.round(result["quantities"], 4))
print("Utility:", np.round(result["utility"], 4))

Utility Testing

In [ ]:
def CES_n_goods_utility_test():
    test_cases = [
        {"alpha": [0.4, 0.3, 0.3], "rho": 0.5, "budget": 100},
        {"alpha": [0.5, 0.3, 0.2], "rho": 0.5, "budget": 100},
        {"alpha": [1/3, 1/3, 1/3], "rho": 0.5, "budget": 100},
        {"alpha": [0.7, 0.2, 0.1], "rho": 0.5, "budget": 100},
        {"alpha": [0.9, 0.05, 0.05], "rho": 0.5, "budget": 100},
    ]

    print("Normal Cases")

    for i, case in enumerate(test_cases, 1):
        print(f"\n--- Test Case {i} ---")
        result = maximize_ces_n_goods(case["alpha"], case["rho"], case["budget"])
        print("Quantities:", np.round(result["quantities"], 4))
        print("Utility:", np.round(result["utility"], 4))

CES_n_goods_utility_test()

Normal Cases

--- Test Case 1 ---
Quantities: [47.0585 26.4708 26.4708]
Utility: 34.0

--- Test Case 2 ---
Quantities: [65.7922 23.682  10.5258]
Utility: 38.0

--- Test Case 3 ---
Quantities: [33.3333 33.3333 33.3333]
Utility: 33.3333

--- Test Case 4 ---
Quantities: [90.7402  7.4074  1.8524]
Utility: 54.0

--- Test Case 5 ---
Quantities: [99.3876  0.3085  0.3039]
Utility: 81.5


## 3.0 Variable Defining ##

### 3.1 Number of Voters and Policies

In [127]:
#How many voters?
num_voters = 4
print("Number of voters: ", num_voters)

#How many policies?
num_policies = 3
print("Number of policies: ", num_policies)

rho = 0.5

print()

Number of voters:  4
Number of policies:  3



### 3.2 Expertise and Preference Generation

In [128]:
#Voter expertise by policy (measured on a scale of 1-5)
expertise_matrix = pd.DataFrame(
    prng.integers(1, 6, size = (num_voters, num_policies)),
    columns = [f"policy_{i}" for i in range(num_policies)],
    index = [f"voter_{i}" for i in range(num_voters)])
print("Voter expertise by policy:")
print(expertise_matrix)
#This generates a dataframe with num_voters rows and num_policies columns, each with a prng between 1-5.
#This is meant to indicate how much expertise each voter has on each policy

simple_voter_expertise = np.zeros(num_voters) #simplifies our expertise matrix, which used to be by policy, into a single value
for i in range(num_voters):
    simple_voter_expertise[i] = np.mean(expertise_matrix.iloc[i])
print(simple_voter_expertise)
#We could use a prng to assign single expertise values and not bother at all with per-policy expertise, since how PPV is designed, you either delegate to a voter i or you delegate your vote to a policy, but cannot delegate to voter i for a specific policy; voter i also can exercise liquid democracy.
#So in this regard per-policy expertise doesn't really matter, but I am keeping the code here in case I need it in the future.


print()

#Voter preference by policy
preference_matrix = pd.DataFrame(
    prng.integers(1, 6, size = (num_voters, num_policies)),
    columns = [f"policy_{i}" for i in range(num_policies)],
    index = [f"voter_{i}" for i in range(num_voters)])
print("Voter preference by policy:")
print(preference_matrix)
#Similar to expertise, each voter has a preference for each policy, their policy position measured on a numerical scale of 1-5 scale (for example, 1=highly disapprove 5=highly approve, or it could be the other way around).
#As the code is written right now, the 1-5 values do not need to be ordinal on a single spectrum, as we have yet to implement whether or not a policy gets implemented (1-2=vote no, 3=vote neutral, 4-5=vote yes). However, it does seem odd that someone would delegate their vote to an issue only to vote neutral, so maybe when we do do yes/no voting it shouldbe on a scale of 1-4?
#However, if this is changed to 1-4, then we have a problem where since the values of expertise and preference are ordinal, and right now they are compared the same (i.e. vote delegation = expertise + preference), expertise would have a higher ceiling than preference (max of 5 vs 4 respectively). This could be solved by using negative numbers. 

def matrix_value_call(voter, policy, called_matrix): #calls specific values within any matrix. Note: this is zero indexed
    return called_matrix.iloc[voter, policy]

matrix_value_call(2,1,preference_matrix) #testing

Voter expertise by policy:
         policy_0  policy_1  policy_2
voter_0         3         5         2
voter_1         1         1         1
voter_2         3         4         5
voter_3         4         2         2
[3.33333333 1.         4.         2.66666667]

Voter preference by policy:
         policy_0  policy_1  policy_2
voter_0         2         3         4
voter_1         5         1         5
voter_2         1         4         1
voter_3         3         5         4


4

### 3.3 Calculating Preference Alignment

In [129]:
#This code generates a three-dimensional matrix to store policy alignment between different voters for different policies, with the axes voter i, voter j, and policy p respectively.
def complete_preference_calculator ():
    complete_preference_matrix = np.zeros((num_voters, num_voters, num_policies))
    for p in range(num_policies):
        for i in range(num_voters):
            for j in range(num_voters):
                complete_preference_matrix[i, j, p] = 5 - np.abs(matrix_value_call(i, p, preference_matrix) - (matrix_value_call(j,p,preference_matrix))) #for a specific policy, if the alignment is exact, then value = 5. If it differs by 1 (i.e. 2 vs 3) then the alignment is 4.If they differ completely (1 vs 5) then the alignment is 1.
    return complete_preference_matrix

complete_preference_alignment = complete_preference_calculator()

print(preference_matrix)
print(complete_preference_alignment)
print(complete_preference_alignment [0,2,0])

def simple_preference_calculator (big_pref_matrix): #takes the complete preference alignment matrix, which measures i x j preference alignment for every policy, and averages the values to generate a single number for i  j prefernce alignment
    simple_preference_matrix = np.zeros((num_voters, num_voters))
    for i in range(num_voters):
        for j in range(num_voters):
            simple_preference_matrix[i, j] = np.mean(big_pref_matrix[i, j, :])
    return simple_preference_matrix

simple_preference_alignment = simple_preference_calculator (complete_preference_alignment)

print(simple_preference_alignment)

         policy_0  policy_1  policy_2
voter_0         2         3         4
voter_1         5         1         5
voter_2         1         4         1
voter_3         3         5         4
[[[5. 5. 5.]
  [2. 3. 4.]
  [4. 4. 2.]
  [4. 3. 5.]]

 [[2. 3. 4.]
  [5. 5. 5.]
  [1. 2. 1.]
  [3. 1. 4.]]

 [[4. 4. 2.]
  [1. 2. 1.]
  [5. 5. 5.]
  [3. 4. 2.]]

 [[4. 3. 5.]
  [3. 1. 4.]
  [3. 4. 2.]
  [5. 5. 5.]]]
4.0
[[5.         3.         3.33333333 4.        ]
 [3.         5.         1.33333333 2.66666667]
 [3.33333333 1.33333333 5.         3.        ]
 [4.         2.66666667 3.         5.        ]]


## 4.0 Voter to Voter Delegation

### 4.1 Voter j Delegates to Voter i

Here, we have the function that allows a single voter j to delegate to all voter i.

In [130]:
#for voter j, delegate to all voter i. When this is completed, run for all voter js.

def j_delegates_to_i (j, rho): #If we really wanted to be picky, then we could make every j voter have a different rho; if not we could just define it globally.
    #the CES function has inputs of alpha, rho, and budget. Rho = whatever, budget = 100. Thus alpha must be defined

    alpha_list = np.zeros(num_voters) #Since there are as many alpha values as goods/i voters (delegatees), we start by generating a blank list of n length
    for i in range(len(alpha_list)):
        alpha_list[i] = (simple_preference_alignment[i, j] + simple_voter_expertise[i]) #calculates the alpha of each voter i, or in other words, the attractiveness of delegating to them, which is a sum of preference alignment and expertise.
    alpha_list = alpha_list/np.sum(alpha_list) #Since our CES funcion requires that all alpha sum to 1, this normalizes our values
    return maximize_ces_n_goods (alpha_list, rho, 100) #maximizes CES function with alpha values (Preference alignment + expertise), rho (some value), and budget=100

### 4.2 Stacking All Voter j to Voter i Delegations Into a Single Matrix

Using the code from 4.1, we iterate the function across all voter j, thus having all voters delegate to all voters, completing our voter to voter delegation stage. If the function from 4.1 produces a list, here we stack all of the lists to output a matrix.

In [131]:
def voter_to_voter_delegations (rho):
    delegation_utility_list = np.zeros(num_voters) #empty utility list
    n_x_n_matrix = np.array(j_delegates_to_i (0, rho)['quantities']).reshape(1, -1) #because we are using numpy array matrix, we have to first calculate the first j and then add onto that
    delegation_utility_list [0] = j_delegates_to_i (0, rho)['utility'] #Here we calculate the first j's delegations and utility

    for j in range(1, num_voters):
        new_row = np.array(j_delegates_to_i(j, rho)['quantities']).reshape(1, -1)
        n_x_n_matrix = np.vstack((n_x_n_matrix, new_row))
        delegation_utility_list[j] = j_delegates_to_i(j, rho)['utility']
    return {
            'delegation_matrix': n_x_n_matrix,
            'utility': delegation_utility_list
        }

print(voter_to_voter_delegations(0.5)['delegation_matrix'])
print(voter_to_voter_delegations(0.5)['utility'])

[[37.80705073  8.71170586 29.2744357  24.20680771]
 [30.06234688 27.20281631 21.36741597 21.36742084]
 [27.26424675  3.34019985 49.69402423 19.70152917]
 [30.76717277  7.68276514 27.98007739 33.56998469]]
[26.48613976 25.14171303 29.10136875 26.56434025]


In [132]:
voter_to_voter_matrix = np.transpose(voter_to_voter_delegations(rho)['delegation_matrix']) #This calls the function from section 4 and assigns it to a variable. Nothing special.
#However, we may need to globally define rho in the future, when running the entire code all at once.

voter_to_voter_utility = voter_to_voter_delegations(rho)['utility']

print(voter_to_voter_matrix)

[[37.80705073 30.06234688 27.26424675 30.76717277]
 [ 8.71170586 27.20281631  3.34019985  7.68276514]
 [29.2744357  21.36741597 49.69402423 27.98007739]
 [24.20680771 21.36742084 19.70152917 33.56998469]]


## 5.0 Voter to Policy Delegation

- First, we need to create our final matrix as portrayed in the paper, with the four quadrants.
- Second, we can put the results from section 4.0 in quadrant 1
- Third, we use the self-delegations (voter 1 to voter 1, voter 2 to voter 2, i.e. the diagonal axis on our 4.0 axis) to select policies based on policy preferences. Put this in quadrant 3.
- Fourth, we fill out quadrant 2 and 4

### 5.1 Voter j Delegates to Policy p

In [133]:
#We go through the same methodology, first having each voter delegate to policies, and then afterwards stacking this all up into a matrix

def j_delegates_to_p (j, rho): #Similar framework for j_delegates_to_i, but with policy factors
    alpha_list = preference_matrix.iloc[j] #Our alphas for voter j are just their policy preferences for all policy p, which is already contained in voter j's row in our preference_matrix
    alpha_list = alpha_list/np.sum(alpha_list) #Since our CES funcion requires that all alpha sum to 1, this normalizes our values
    return maximize_ces_n_goods (alpha_list, rho, 100) #maximizes CES function with alpha values, rho (some value), and budget=100

### 5.2 Stacking All Voter j to Policy p Delegations Into a Single Matrix

In [140]:
def voter_to_policy_delegations (rho):
    delegation_utility_list = np.zeros(num_voters) #empty utility list
    n_x_p_matrix = np.array(j_delegates_to_p (0, rho)['quantities']).reshape(1, -1) #First j's policy delegations
    delegation_utility_list [0] = j_delegates_to_p (0, rho)['utility'] #Here we calculate the first j's delegations and utility

    for j in range(1, num_voters):
        new_row = np.array(j_delegates_to_p(j, rho)['quantities']).reshape(1, -1)
        n_x_p_matrix = np.vstack((n_x_p_matrix, new_row))
        delegation_utility_list[j] = j_delegates_to_p(j, rho)['utility']
    return {
            'delegation_matrix': n_x_p_matrix,
            'utility': delegation_utility_list
        }

print(voter_to_policy_delegations(0.5)['delegation_matrix'])
print(voter_to_policy_delegations(0.5)['utility'])

[[13.79302115 31.02847565 55.1785032 ]
 [49.01952095  1.9609581  49.01952095]
 [ 5.55551045 88.88897815  5.5555114 ]
 [17.99988679 50.00009022 32.00002299]]
[35.80246897 42.14876033 50.         34.72222222]


In [41]:
voter_to_policy_matrix = np.transpose(voter_to_policy_delegations(rho)['delegation_matrix'])

voter_to_policy_utility = voter_to_policy_delegations(rho)['utility']

print(voter_to_policy_matrix)

[[33.33333333 32.00004196 35.54642933]
 [33.33333333 50.00006781  8.88911485]
 [33.33333333 17.99989023 55.56445582]]


## 6.0 Creation of the Full Delegation Matrix

### 6.1 Creating Policy to Voter Matrix (Zero) and Policy to Policy Matrix (Identity)

In [98]:
policy_to_voter_matrix = np.zeros((num_voters, num_policies))
policy_to_policy_matrix = np.identity((num_policies))

### 6.2 Compiling the Full Delegation Matrix (Raw)

In [99]:
full_delegation_matrix = np.block([
    [voter_to_voter_matrix, policy_to_voter_matrix],
    [voter_to_policy_matrix, policy_to_policy_matrix] 
])

def normalize_full_delegation_matrix():
    for j in range(num_voters):
        for p in range(num_voters, num_voters+num_policies):
            full_delegation_matrix[p, j] = full_delegation_matrix[p, j]/100 * full_delegation_matrix[j,j]
        full_delegation_matrix[j, j] = 0

normalize_full_delegation_matrix()

print(full_delegation_matrix)

[[ 0.         23.83647638 21.78142647  0.          0.          0.        ]
 [36.82721996  0.         31.36352126  0.          0.          0.        ]
 [29.82492726 26.90488101  0.          0.          0.          0.        ]
 [11.11595093 15.7627863  16.65529804  1.          0.          0.        ]
 [11.11595093 24.62935471  4.16499941  0.          1.          0.        ]
 [11.11595093  8.8665016  26.03475482  0.          0.          1.        ]]


## 8.0 Ex Post Utility Calculation

Utility is calculated in 4 distinct ways:
- Voter to voter delegation stage: expertise and preference alignment (variable: voter_to_voter_utility)
- Voter to policy ex ante: voters selecting policy (variable: voter_to_policy_utility)
- Voter to policy ex post: voter's policy alignment x actual voting outcome (TBD)
- Control Loss Cost: excessive delegation (TBD)